<a href="https://colab.research.google.com/github/thalesfreirefarias/Controle-Financeiro-Pro-/blob/main/02_movimentacoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ==========================================
# CONTROLE FINANCEIRO PRO 2.0
# DIA 2 - MOVIMENTAÇÕES FINANCEIRAS
# ==========================================


# 1. Importação das bibliotecas

import pandas as pd
import requests
import base64

from google.colab import userdata


# 2. Criação da base financeira

dados = {
    "Data": [
        "05/11/2026", "06/11/2026", "08/11/2026",
        "10/11/2026", "15/11/2026", "20/11/2026"
    ],

    "Descrição": [
        "Salário", "Supermercado", "Restaurante",
        "Aluguel", "Freelance", "Internet"
    ],

    "Tipo": [
        "Entrada", "Saída", "Saída",
        "Saída", "Entrada", "Saída"
    ],

    "Categoria": [
        "Salário", "Alimentação", "Lazer",
        "Moradia", "Renda Extra", "Serviços"
    ],

    "Pagamento": [
        "Transferência", "Pix", "Crédito",
        "Pix", "Transferência", "Débito"
    ],

    "Valor": [
        5000.00, 250.00, 150.00,
        1500.00, 800.00, 120.00
    ]
}

df = pd.DataFrame(dados)


# 3. Tratamento dos dados

df["Data"] = pd.to_datetime(
    df["Data"],
    format="%d/%m/%Y"
)

display(df)


# 4. Qualidade dos dados

print("Dimensão da base:", df.shape)

print("\nTipos de dados:")
print(df.dtypes)

print("\nValores nulos:")
print(df.isnull().sum())

assert df.notna().all().all(), "Existem valores nulos"

assert df["Valor"].gt(0).all(), "Valores inválidos"

assert df["Tipo"].isin(
    ["Entrada", "Saída"]
).all(), "Tipos inválidos"

print("\nValidação concluída com sucesso!")


# 5. Indicadores financeiros

receitas = df.loc[
    df["Tipo"] == "Entrada", "Valor"
].sum()

despesas = df.loc[
    df["Tipo"] == "Saída", "Valor"
].sum()

saldo = receitas - despesas

economia = saldo / receitas * 100 if receitas > 0 else 0

print("\nINDICADORES FINANCEIROS")

print(f"Receitas: R$ {receitas:.2f}")
print(f"Despesas: R$ {despesas:.2f}")
print(f"Saldo: R$ {saldo:.2f}")
print(f"Capacidade de economia: {economia:.2f}%")


# 6. Despesas por categoria

despesas_categoria = (
    df[df["Tipo"] == "Saída"]
    .groupby("Categoria")["Valor"]
    .sum()
    .sort_values(ascending=False)
)

print("\nDESPESAS POR CATEGORIA")

display(despesas_categoria)


# 7. Configuração da conexão com o GitHub

token = userdata.get("GITHUB_TOKEN")

repositorio = "thalesfreirefarias/Controle-Financeiro-Pro-"

caminho = "data/movimentacoes.csv"

url = (
    f"https://api.github.com/repos/"
    f"{repositorio}/contents/{caminho}"
)

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json"
}


# 8. Conversão do DataFrame em CSV

csv = df.to_csv(index=False)

conteudo = base64.b64encode(
    csv.encode("utf-8-sig")
).decode("utf-8")


# 9. Verificação do arquivo no GitHub

consulta = requests.get(
    url,
    headers=headers,
    params={"ref": "main"},
    timeout=30
)

payload = {
    "message": "feat: publish financial dataset",
    "content": conteudo,
    "branch": "main"
}

if consulta.status_code == 200:
    payload["sha"] = consulta.json()["sha"]

elif consulta.status_code != 404:
    consulta.raise_for_status()


# 10. Publicação automática no GitHub

resposta = requests.put(
    url,
    headers=headers,
    json=payload,
    timeout=30
)

resposta.raise_for_status()

print("\nCSV publicado no GitHub com sucesso!")

print("Endereço do arquivo:")
print(resposta.json()["content"]["html_url"])

,Data,Descrição,Tipo,Categoria,Pagamento,Valor
0,2026-11-05,Salário,Entrada,Salário,Transferência,5000.0
1,2026-11-06,Supermercado,Saída,Alimentação,Pix,250.0
2,2026-11-08,Restaurante,Saída,Lazer,Crédito,150.0
3,2026-11-10,Aluguel,Saída,Moradia,Pix,1500.0
4,2026-11-15,Freelance,Entrada,Renda Extra,Transferência,800.0
5,2026-11-20,Internet,Saída,Serviços,Débito,120.0


Dimensão da base: (6, 6)

Tipos de dados:
Data         datetime64[ns]
Descrição            object
Tipo                 object
Categoria            object
Pagamento            object
Valor               float64
dtype: object

Valores nulos:
Data         0
Descrição    0
Tipo         0
Categoria    0
Pagamento    0
Valor        0
dtype: int64

Validação concluída com sucesso!

INDICADORES FINANCEIROS
Receitas: R$ 5800.00
Despesas: R$ 2020.00
Saldo: R$ 3780.00
Capacidade de economia: 65.17%

DESPESAS POR CATEGORIA


,Valor
Categoria,
Moradia,1500.0
Alimentação,250.0
Lazer,150.0
Serviços,120.0



CSV publicado no GitHub com sucesso!
Endereço do arquivo:
https://github.com/thalesfreirefarias/Controle-Financeiro-Pro-/blob/main/data/movimentacoes.csv


In [2]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

print("Token configurado com sucesso!" if token else "Erro")

Token configurado com sucesso!
